# Interactive DST Explorer

Enter a **prompt**, **model name**, **context words**, and **noncontext words** below to run full Distributional Semantics Tracing with both CAS methods (Probability & Cosine).

| CAS Method | Measures | Best for |
|---|---|---|
| **Probability** (softmax) | Which words the model will *predict* at each layer | Misalignment / refusal analysis |
| **Cosine** (abs cosine sim) | How much *information* about each word set is encoded | Hallucination / context retention |

## 1. Setup

In [ ]:
import sys, os, warnings
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from transformers import AutoModelForCausalLM, AutoTokenizer
from ltr.dst import (
    DistributionalSemanticsTracer,
    DSTResult,
    SemanticMap,
    CASTrace,
)

try:
    import networkx as nx
except ImportError:
    raise ImportError("networkx is required: pip install networkx")

if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

## 2. Input Configuration

Edit the variables below and re-run from this cell onward.

In [ ]:
# ====================== USER INPUTS ======================

MODEL_NAME = "Qwen/Qwen3-0.6B"  # Any HuggingFace causal LM

PROMPT = "The man went to the bank by the river to deposit his"

CONTEXT_WORDS = ["river", "water", "shore", "stream", "fish", "nature"]

NONCONTEXT_WORDS = ["money", "finance", "account", "loan", "credit", "bank"]

TOP_K = 15  # Number of concept nodes per layer

# ==========================================================

print(f"Model:            {MODEL_NAME}")
print(f"Prompt:           {PROMPT!r}")
print(f"Context words:    {CONTEXT_WORDS}")
print(f"Noncontext words: {NONCONTEXT_WORDS}")
print(f"Top-K:            {TOP_K}")

## 3. Load Model & Tokenizer

In [ ]:
print(f"Loading model: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32,
).to(DEVICE)
model.eval()

# Detect layer prefix
cfg = model.config
n_layers = getattr(cfg, "num_hidden_layers", getattr(cfg, "n_layer", 12))
d_model = getattr(cfg, "hidden_size", getattr(cfg, "n_embd", None))

# Auto-detect layer prefix
LAYER_PREFIX = "model.layers."
for prefix_candidate in ["model.layers.", "transformer.h.", "gpt_neox.layers."]:
    try:
        _ = dict(model.named_modules())
        first_layer = prefix_candidate + "0"
        if first_layer in dict(model.named_modules()):
            LAYER_PREFIX = prefix_candidate
            break
    except Exception:
        pass

print(f"Layers: {n_layers}  |  Width: {d_model}  |  Prefix: {LAYER_PREFIX}")
print("Model loaded.")

## 4. Build DST Tracer & Run Both CAS Methods

In [ ]:
tracer = DistributionalSemanticsTracer(
    model, tokenizer, device=DEVICE, layer_prefix=LAYER_PREFIX
)

print("Running DST with Probability CAS ...")
result_prob = tracer.run_analysis(
    prompt=PROMPT,
    context_words=CONTEXT_WORDS,
    noncontext_words=NONCONTEXT_WORDS,
    K=TOP_K,
    compute_edges=True,
    cas_method="probability",
)

print("Running DST with Cosine CAS ...")
result_cos = tracer.run_analysis(
    prompt=PROMPT,
    context_words=CONTEXT_WORDS,
    noncontext_words=NONCONTEXT_WORDS,
    K=TOP_K,
    compute_edges=True,
    cas_method="cosine",
)

print(f"\nGenerated text: {result_prob.generated_text!r}")
print("Done.")

## 5. CAS Traces — Probability vs Cosine (Side by Side)

In [ ]:
fig, (ax_p, ax_c) = plt.subplots(1, 2, figsize=(16, 5))

for ax, cas_trace, method_name, color in [
    (ax_p, result_prob.cas_trace, "Probability CAS", "#2d3436"),
    (ax_c, result_cos.cas_trace, "Cosine CAS", "#6c5ce7"),
]:
    layers = list(range(len(cas_trace.cas_values)))
    ax.plot(layers, cas_trace.cas_values, "o-", color=color, linewidth=2, markersize=4)
    ax.axhline(0.5, color="grey", linestyle=":", alpha=0.7, label="Neutral (0.5)")
    ax.fill_between(layers, 0.5, 1.0, alpha=0.04, color="green")
    ax.fill_between(layers, 0.0, 0.5, alpha=0.04, color="red")

    if cas_trace.onset_layer is not None:
        ax.plot(cas_trace.onset_layer, cas_trace.cas_values[cas_trace.onset_layer],
                "o", color="green", markersize=14, zorder=5, label="Onset")
    if cas_trace.inversion_layer is not None:
        ax.plot(cas_trace.inversion_layer, cas_trace.cas_values[cas_trace.inversion_layer],
                "o", color="#f1c40f", markersize=14, zorder=5, label="Inversion")
    if cas_trace.commitment_layer is not None:
        ax.plot(cas_trace.commitment_layer, cas_trace.cas_values[cas_trace.commitment_layer],
                "o", color="red", markersize=14, zorder=5, label="Commitment")

    ax.set_xlabel("Layer")
    ax.set_ylabel("CAS")
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f"{method_name}\n{PROMPT[:60]}...", fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. CAS Traces Overlaid

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

layers = list(range(len(result_prob.cas_trace.cas_values)))
ax.plot(layers, result_prob.cas_trace.cas_values, "o-", color="steelblue",
        linewidth=2, markersize=4, label="Probability CAS")
ax.plot(layers, result_cos.cas_trace.cas_values, "s--", color="#e17055",
        linewidth=2, markersize=4, label="Cosine CAS")
ax.axhline(0.5, color="grey", linestyle=":", alpha=0.7)
ax.fill_between(layers, 0.5, 1.0, alpha=0.03, color="green")
ax.fill_between(layers, 0.0, 0.5, alpha=0.03, color="red")

# Markers for probability CAS
ct = result_prob.cas_trace
if ct.onset_layer is not None:
    ax.axvline(ct.onset_layer, color="green", linestyle=":", alpha=0.5, label="Prob onset")
if ct.inversion_layer is not None:
    ax.axvline(ct.inversion_layer, color="#f1c40f", linestyle="-.", alpha=0.5, label="Prob inversion")
if ct.commitment_layer is not None:
    ax.axvline(ct.commitment_layer, color="red", linestyle="--", alpha=0.5, label="Prob commitment")

ax.set_xlabel("Layer")
ax.set_ylabel("CAS")
ax.set_ylim(-0.05, 1.05)
ax.set_title(f"Probability vs Cosine CAS — {PROMPT[:60]}...", fontsize=11)
ax.legend(fontsize=8, loc="best")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Layer Markers Summary

In [ ]:
import pandas as pd

rows = []
for method, res in [("Probability", result_prob), ("Cosine", result_cos)]:
    ct = res.cas_trace
    cas = np.array(ct.cas_values)
    rows.append({
        "CAS Method": method,
        "Onset Layer": ct.onset_layer,
        "Inversion Layer": ct.inversion_layer,
        "Commitment Layer": ct.commitment_layer,
        "Mean CAS": f"{cas.mean():.4f}",
        "Final CAS": f"{cas[-1]:.4f}",
        "Min CAS": f"{cas.min():.4f}",
        "Max CAS": f"{cas.max():.4f}",
    })

df_markers = pd.DataFrame(rows)
display(df_markers)

## 8. Next-Token Probability Distribution

In [ ]:
fig = tracer.plot_next_token_probs(result_prob.next_token_probs, top_n=20)
plt.title(f"Next-token probabilities — {PROMPT[:50]}...")
plt.tight_layout()
plt.show()

## 9. Semantic Maps at Key Layers (Probability CAS)

In [ ]:
fig = tracer.plot_layer_maps_grid(
    result_prob,
    context_words=CONTEXT_WORDS,
    noncontext_words=NONCONTEXT_WORDS,
    cols=3,
)
fig.suptitle("Semantic Maps (Probability CAS run)", fontsize=13, y=1.01)
plt.show()

## 10. DST Summary Figure (Probability CAS)

In [ ]:
fig = tracer.plot_dst_summary(
    result_prob,
    context_words=CONTEXT_WORDS,
    noncontext_words=NONCONTEXT_WORDS,
)
fig.suptitle(f"DST Summary (Probability) — {MODEL_NAME}", fontsize=13, y=1.01)
plt.show()

## 11. DST Summary Figure (Cosine CAS)

In [ ]:
fig = tracer.plot_dst_summary(
    result_cos,
    context_words=CONTEXT_WORDS,
    noncontext_words=NONCONTEXT_WORDS,
)
fig.suptitle(f"DST Summary (Cosine) — {MODEL_NAME}", fontsize=13, y=1.01)
plt.show()

## 12. Concept Scores Distribution (Early / Mid / Late)

In [ ]:
# Extract residual stream for the prompt
@torch.no_grad()
def extract_residual_stream(model, tokenizer, prompt, device=DEVICE):
    # Apply chat template if available
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            formatted = tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False, add_generation_prompt=True,
            )
        except Exception:
            formatted = prompt
    else:
        formatted = prompt
    inputs = tokenizer(formatted, return_tensors="pt").to(device)
    outputs = model(**inputs, output_hidden_states=True)
    answer_pos = inputs["input_ids"].shape[1] - 1
    hs = torch.stack([h[0, answer_pos] for h in outputs.hidden_states])
    return hs

# Get unembedding matrix
if hasattr(model, "lm_head"):
    U = model.lm_head.weight.detach().to(DEVICE)
elif hasattr(model, "get_output_embeddings"):
    U = model.get_output_embeddings().weight.detach().to(DEVICE)
else:
    U = model.get_input_embeddings().weight.detach().to(DEVICE)

hs = extract_residual_stream(model, tokenizer, PROMPT)
concept_scores = hs @ U.T  # (L+1, |V|)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
layer_picks = [1, n_layers // 2, n_layers]

for ax, l in zip(axes, layer_picks):
    scores_np = concept_scores[l].cpu().numpy()
    ax.hist(scores_np, bins=100, color="#74b9ff", edgecolor="white", linewidth=0.3)
    top5 = concept_scores[l].topk(5)
    top5_words = [tokenizer.decode([tid]).strip() for tid in top5.indices.tolist()]
    ax.set_title(f"Layer {l} — top-5: {', '.join(top5_words)}", fontsize=10)
    ax.set_xlabel("Concept score s(v)")

axes[0].set_ylabel("Count")
fig.suptitle("Concept-score distributions sharpen across depth", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 13. Top Causal Edges by Layer

In [ ]:
print(f"{'Layer':>5}  {'Source':>18} → {'Target':<18}  {'Weight':>10}  {'Pos':>4}")
print("-" * 65)

for layer in sorted(result_prob.semantic_maps.keys()):
    sm = result_prob.semantic_maps[layer]
    for e in sm.edges[:3]:  # top 3 per layer
        print(f"{layer:5d}  {e.source:>18} → {e.target:<18}  {e.weight:+10.5f}  {e.source_position:4d}")

## 14. Per-Layer CAS Values Table

In [ ]:
cas_p = result_prob.cas_trace.cas_values
cas_c = result_cos.cas_trace.cas_values

cas_rows = []
for i in range(len(cas_p)):
    markers = []
    for method, ct in [("P", result_prob.cas_trace), ("C", result_cos.cas_trace)]:
        if i == ct.onset_layer:
            markers.append(f"{method}-Onset")
        if i == ct.inversion_layer:
            markers.append(f"{method}-Inversion")
        if i == ct.commitment_layer:
            markers.append(f"{method}-Commitment")
    cas_rows.append({
        "Layer": i,
        "Prob CAS": f"{cas_p[i]:.4f}",
        "Cosine CAS": f"{cas_c[i]:.4f}",
        "Markers": ", ".join(markers) if markers else "",
    })

df_cas = pd.DataFrame(cas_rows)
display(df_cas)

## 15. Misalignment Drift Score (MDS)

In [ ]:
for method, res in [("Probability", result_prob), ("Cosine", result_cos)]:
    cas = np.array(res.cas_trace.cas_values)
    drift_mask = cas < 0.5
    mds = float((0.5 - cas[drift_mask]).mean()) if drift_mask.any() else 0.0
    n_drift = int(drift_mask.sum())
    print(f"{method} CAS:")
    print(f"  MDS (mean drift below 0.5): {mds:.4f}")
    print(f"  Layers below neutral:       {n_drift}/{len(cas)}")
    print(f"  Final CAS:                  {cas[-1]:.4f}")
    print()

## 16. Concept Node Details at Selected Layers

In [ ]:
ctx_set = set(CONTEXT_WORDS)
nonctx_set = set(NONCONTEXT_WORDS)

for layer in sorted(result_prob.semantic_maps.keys()):
    sm = result_prob.semantic_maps[layer]
    print(f"\n--- Layer {layer}: {len(sm.nodes)} nodes, {len(sm.edges)} edges ---")
    for n in sm.nodes[:10]:
        tag = ""
        if n.word.lower() in ctx_set or n.word in ctx_set:
            tag = " [CTX]"
        elif n.word.lower() in nonctx_set or n.word in nonctx_set:
            tag = " [NONCTX]"
        print(f"  {n.word:20s}  score={n.score:+8.3f}  affinity={n.affinity:+.4f}{tag}")